In [ ]:
import socket
import json
import time
import math
import uuid
import os
import numpy as np

# ── Network ───────────────────────────────────────────────────────────────────
LAPTOP_IP         = "192.168.2.4"   # laptop ethernet IP
LAPTOP_SHAPE_PORT = 5006            # port laptop listens on for recognised shapes
PYNQ_LISTEN_PORT  = 5005            # port this node listens on for strokes

# ── Shape recognition ────────────────────────────────────────────────────────
RESAMPLE_STEP      = 8.0    # pixels between resampled points
RDP_EPSILON        = 12.0   # RDP simplification tolerance (pixels)
CLOSE_THRESH_PX    = 60.0   # max distance first-to-last point to count as closed
CIRCLE_REL_TOL     = 0.18   # max rmse/r ratio for circle classification
RECT_ANGLE_TOL     = 50.0   # degrees tolerance from 90 for rectangle corners

# ── Output directory ─────────────────────────────────────────────────────────
OUTPUT_DIR = "./shapes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration loaded.")
print(f"Listening on port {PYNQ_LISTEN_PORT}")
print(f"Sending shapes to {LAPTOP_IP}:{LAPTOP_SHAPE_PORT}")
print(f"JSON output: {OUTPUT_DIR}")

In [ ]:
# ── Resample polyline ─────────────────────────────────────────────────────────
# PL target: fixed-point linear interpolation pipeline

def resample_polyline(points, step=RESAMPLE_STEP):
    """Resample stroke so consecutive points are ~step pixels apart."""
    if len(points) < 2:
        return points[:]
    out  = [points[0]]
    acc  = 0.0
    prev = np.array(points[0], dtype=float)
    for p in points[1:]:
        cur = np.array(p, dtype=float)
        seg = np.linalg.norm(cur - prev)
        if seg < 1e-6:
            continue
        while acc + seg >= step:
            t    = (step - acc) / seg
            newp = prev + t * (cur - prev)
            out.append((int(round(newp[0])), int(round(newp[1]))))
            prev = newp
            seg  = np.linalg.norm(cur - prev)
            acc  = 0.0
        acc  += seg
        prev  = cur
    if out[-1] != points[-1]:
        out.append(points[-1])
    return out


# ── RDP simplification ────────────────────────────────────────────────────────
# PL target: pipelined iterative perpendicular distance

def rdp_simplify(points, eps=RDP_EPSILON):
    """Ramer-Douglas-Peucker polyline simplification."""
    if len(points) < 3:
        return points
    a   = np.array(points[0],  dtype=float)
    b   = np.array(points[-1], dtype=float)
    ab  = b - a
    ab2 = float(ab @ ab)
    max_d = -1.0
    idx   = -1
    for i in range(1, len(points) - 1):
        p = np.array(points[i], dtype=float)
        if ab2 < 1e-9:
            d = np.linalg.norm(p - a)
        else:
            t    = float(((p - a) @ ab) / ab2)
            proj = a + np.clip(t, 0.0, 1.0) * ab
            d    = np.linalg.norm(p - proj)
        if d > max_d:
            max_d = d
            idx   = i
    if max_d > eps:
        left  = rdp_simplify(points[:idx + 1], eps)
        right = rdp_simplify(points[idx:],     eps)
        return left[:-1] + right
    return [points[0], points[-1]]


# ── Kasa circle fit ───────────────────────────────────────────────────────────
# PL target: fixed-point multiply-accumulate matrix solve

def kasa_circle_fit(points):
    """Fit a circle using the Kasa method. Returns (cx, cy, r, rmse) or None."""
    pts = np.array(points, dtype=float)
    x   = pts[:, 0]
    y   = pts[:, 1]
    A   = np.column_stack([x, y, np.ones_like(x)])
    b   = x * x + y * y
    try:
        c, *_ = np.linalg.lstsq(A, b, rcond=None)
    except np.linalg.LinAlgError:
        return None
    cx   = 0.5 * c[0]
    cy   = 0.5 * c[1]
    r    = np.sqrt(max(1e-9, cx*cx + cy*cy + c[2]))
    d    = np.sqrt((x - cx)**2 + (y - cy)**2)
    rmse = float(np.sqrt(np.mean((d - r)**2)))
    return float(cx), float(cy), float(r), rmse


# ── Rectangle corner angle computation ────────────────────────────────────────
# PL target: parallel angle computation across 4 corners

def angle_deg(u, v):
    u  = np.array(u, dtype=float)
    v  = np.array(v, dtype=float)
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu < 1e-9 or nv < 1e-9:
        return 0.0
    c = float(np.clip((u @ v) / (nu * nv), -1.0, 1.0))
    return float(np.degrees(np.arccos(c)))

def try_rectangle(points, eps=None, right_angle_tol=RECT_ANGLE_TOL):
    """
    Lenient rectangle detector.
    Simplifies with RDP, trims to 4 corners, checks right angles.
    Returns list of 4 corner points or None.
    """
    if eps is None:
        xs    = [p[0] for p in points]
        ys    = [p[1] for p in points]
        scale = max(20.0, max(max(xs) - min(xs), max(ys) - min(ys)))
        eps   = 0.06 * scale

    simp = rdp_simplify(points, eps)
    if len(simp) >= 2 and simp[0] == simp[-1]:
        simp = simp[:-1]

    # trim to 4 corners by dropping shortest edges
    while len(simp) > 4:
        n    = len(simp)
        lens = []
        for i in range(n):
            a = np.array(simp[i],           dtype=float)
            b = np.array(simp[(i+1) % n],   dtype=float)
            lens.append(np.linalg.norm(b - a))
        simp.pop(int(np.argmin(lens)))

    if len(simp) != 4:
        return None

    for i in range(4):
        p_prev = np.array(simp[(i-1) % 4], dtype=float)
        p      = np.array(simp[i],         dtype=float)
        p_next = np.array(simp[(i+1) % 4], dtype=float)
        ang    = angle_deg(p_prev - p, p_next - p)
        if abs(ang - 90.0) > right_angle_tol:
            return None

    return simp

print("All PL stub functions defined.")

In [ ]:
def recognise_shape(stroke_points):
    """
    Full shape recognition pipeline.
    Input:  list of [x, y] stroke points (pixel coordinates)
    Output: shape dict ready to send to laptop:
        {'id': ..., 'timestamp': ..., 'type': 'circle',    'params': {'cx','cy','r'}}
        {'id': ..., 'timestamp': ..., 'type': 'rectangle', 'params': {'corners': [[x,y],...]}}
        {'id': ..., 'timestamp': ..., 'type': 'polyline',  'params': {'points': [[x,y],...]}}
    """
    pts = [(p[0], p[1]) for p in stroke_points]

    if len(pts) < 4:
        return _make_shape('polyline', {'points': pts})

    # step 1 — resample to uniform arc length (PL target)
    resampled = resample_polyline(pts, RESAMPLE_STEP)

    # check if stroke is closed
    closed = (len(resampled) >= 10 and
              math.hypot(resampled[0][0] - resampled[-1][0],
                         resampled[0][1] - resampled[-1][1]) < CLOSE_THRESH_PX)
    if closed and resampled[0] != resampled[-1]:
        resampled.append(resampled[0])

    xs    = [p[0] for p in resampled]
    ys    = [p[1] for p in resampled]
    scale = max(20.0, max(max(xs) - min(xs), max(ys) - min(ys)))

    # step 2 — try circle (Kasa fit, PL target)
    if closed and len(resampled) >= 25:
        fit = kasa_circle_fit(resampled[:-1])
        if fit is not None:
            cx, cy, r, rmse = fit
            rel = rmse / max(1.0, r)
            if 15 <= r <= 0.9 * scale and rel <= CIRCLE_REL_TOL:
                return _make_shape('circle', {
                    'cx': round(float(cx), 2),
                    'cy': round(float(cy), 2),
                    'r':  round(float(r),  2),
                })

    # step 3 — try rectangle (corner angle computation, PL target)
    if closed:
        corners = try_rectangle(resampled)
        if corners is not None:
            return _make_shape('rectangle', {
                'corners': [[round(float(p[0]), 2), round(float(p[1]), 2)] for p in corners]
            })

    # step 4 — fallback: RDP-simplified polyline (PL target)
    simplified = rdp_simplify(pts, RDP_EPSILON)
    return _make_shape('polyline', {
        'points': [[round(float(p[0]), 2), round(float(p[1]), 2)] for p in simplified]
    })


def _make_shape(stype, params):
    return {
        'id':        str(uuid.uuid4())[:8],
        'timestamp': time.time(),
        'type':      stype,
        'params':    params,
    }

print("recognise_shape defined.")

In [ ]:
def save_shape_json(shape):
    """Save a completed shape to /home/xilinx/shapes/<id>.json"""
    path = os.path.join(OUTPUT_DIR, f"{shape['id']}.json")
    with open(path, 'w') as f:
        json.dump(shape, f, indent=2)
    print(f"[json] saved {path}")

print("save_shape_json defined.")

In [ ]:
def benchmark_pipeline(stroke_points, runs=100):
    """
    Time each stage of the pipeline individually over `runs` iterations.
    Call this from a separate cell to measure software baseline latency.
    """
    pts = [(p[0], p[1]) for p in stroke_points]

    # resample
    t0 = time.perf_counter()
    for _ in range(runs):
        r = resample_polyline(pts, RESAMPLE_STEP)
    t_resample = (time.perf_counter() - t0) / runs * 1000

    # RDP
    t0 = time.perf_counter()
    for _ in range(runs):
        s = rdp_simplify(r, RDP_EPSILON)
    t_rdp = (time.perf_counter() - t0) / runs * 1000

    # Kasa
    t0 = time.perf_counter()
    for _ in range(runs):
        kasa_circle_fit(r)
    t_kasa = (time.perf_counter() - t0) / runs * 1000

    # rectangle
    t0 = time.perf_counter()
    for _ in range(runs):
        try_rectangle(r)
    t_rect = (time.perf_counter() - t0) / runs * 1000

    print(f"Software baseline ({runs} runs each):")
    print(f"  Resample polyline : {t_resample:.4f} ms")
    print(f"  RDP simplification: {t_rdp:.4f} ms")
    print(f"  Kasa circle fit   : {t_kasa:.4f} ms")
    print(f"  Rectangle detect  : {t_rect:.4f} ms")
    print(f"  Total pipeline    : {t_resample+t_rdp+t_kasa+t_rect:.4f} ms")

print("benchmark_pipeline defined.")

In [ ]:
test_stroke = [[100+i*3, 200+i*2] for i in range(50)]
benchmark_pipeline(test_stroke)

In [ ]:
recv_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
recv_sock.bind(('0.0.0.0', PYNQ_LISTEN_PORT))
recv_sock.settimeout(1.0)

send_sock   = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
laptop_addr = (LAPTOP_IP, LAPTOP_SHAPE_PORT)

print(f"[pynq_a] listening for strokes on port {PYNQ_LISTEN_PORT}")
print(f"[pynq_a] sending shapes to {LAPTOP_IP}:{LAPTOP_SHAPE_PORT}")
print("[pynq_a] running — stop with the Jupyter stop button")

stroke_count = 0

try:
    while True:
        try:
            data, addr = recv_sock.recvfrom(65535)
            print("[pynq] packet received from", addr)
        except socket.timeout:
            continue

        try:
            packet = json.loads(data.decode('utf-8'))
        except json.JSONDecodeError as e:
            print(f"[pynq_a] bad packet: {e}")
            continue

        stroke_points = packet.get('stroke', [])
        if len(stroke_points) < 2:
            print("[pynq_a] stroke too short, skipping")
            continue

        t_start = time.perf_counter()

        shape = recognise_shape(stroke_points)
        save_shape_json(shape)

        t_ms = (time.perf_counter() - t_start) * 1000

        # send shape back to laptop
        payload = json.dumps(shape).encode('utf-8')
        print("Sending shape to laptop: ", laptop_addr)
        send_sock.sendto(payload, laptop_addr)

        stroke_count += 1
        print(f"[pynq_a] stroke {stroke_count}: {shape['type']:12s} recognised in {t_ms:.2f} ms")

except KeyboardInterrupt:
    print("[pynq_a] stopped.")
finally:
    recv_sock.close()
    send_sock.close()
    print("[pynq_a] sockets closed.")